<a href="https://colab.research.google.com/github/DrYasirSaeed/Shariah-Complaince-Research/blob/main/Shariah_Compliance_Variable_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Paths
file_path = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/2005-23.xlsx'
out_csv = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2005_08.csv'

# --- Helper Functions ---
def normalize_item(s):
    if pd.isna(s): return ""
    return re.sub(r'\s+', ' ', str(s)).strip()

def parse_num(s):
    if pd.isna(s): return None
    t = str(s).strip()
    if t in ("-", "N/A", ""): return None

    neg = False
    if t.startswith('(') and t.endswith(')'):
        neg = True
        t = t[1:-1].strip()

    is_percent = '%' in t
    t = t.replace(',', '').replace('%', '')

    try:
        n = float(t)
        if neg: n = -n
        if is_percent: n = n / 100.0
        return n
    except ValueError:
        return None

def safe_div(a, b):
    if a is None or b is None or pd.isna(a) or pd.isna(b) or b == 0.0:
        return None
    return a / b

def safe_add(*args):
    """Replicates PowerShell's behavior of treating nulls as 0 during addition."""
    valid = [a for a in args if a is not None and not pd.isna(a)]
    if not valid: return None
    return sum(valid)

def get_item_year_value(company_map, item_name, year):
    return company_map.get(item_name, {}).get(year, None)

# --- Execution ---
print("Loading Excel file into memory (Sheet: 2005-08)...")
df = pd.read_excel(file_path, sheet_name='2005-08', header=None)

# Map years to their corresponding zero-indexed column numbers
years = { '2005': 4, '2006': 5, '2007': 6, '2008': 7 }

data = {}
ordered_companies = []  # List to track exact top-to-bottom sequence
company_metadata = {}   # Dictionary to store Sector and Sub-Sector info

print("Parsing data structure and capturing sector hierarchy...")
for index, row in df.iloc[1:].iterrows():
    sector_raw = row[0]     # Col A: Sector
    subsector_raw = row[1]  # Col B: Sub-Sector
    company_raw = row[2]    # Col C: Company Name
    item_raw = row[3]       # Col D: Item Name

    company = str(company_raw).strip() if pd.notna(company_raw) else ""
    item = normalize_item(item_raw)

    if not company or not item: continue

    # Capture company and its metadata on first encounter
    if company not in data:
        data[company] = {}
        ordered_companies.append(company)
        company_metadata[company] = {
            'Sector': str(sector_raw).strip() if pd.notna(sector_raw) else "",
            'Sub-Sector': str(subsector_raw).strip() if pd.notna(subsector_raw) else ""
        }

    if item not in data[company]:
        data[company][item] = {}

    for year_label, col_idx in years.items():
        if col_idx < len(row):
            cell_val = row[col_idx]
            if isinstance(cell_val, (int, float)):
                data[company][item][year_label] = float(cell_val)
            else:
                data[company][item][year_label] = parse_num(cell_val)
        else:
            data[company][item][year_label] = None

print("Computing Shariah-compliance variables in original sequence...")
results = []

# Iterating using the sequence list, NOT alphabetical sort
for company in ordered_companies:
    cm = data[company]
    meta = company_metadata[company]

    for yr in years.keys():

        # --- RAW DATA EXTRACTION ---
        A = get_item_year_value(cm, "A. Non-Current Assets (A2+A3)", yr)
        B = get_item_year_value(cm, "B. Current Assets (B1+B2+B3+B4+B5)", yr)
        if B is None:
            B = get_item_year_value(cm, "B. Current Assets (B1+B2+B3+B4+B5+B6)", yr)

        TotalAB = get_item_year_value(cm, "Total Assets (A+B) / Equity & Liabilities (C+D+E)", yr)
        if TotalAB is None:
            TotalAB = safe_add(A, B) if (A is not None or B is not None) else None

        D = get_item_year_value(cm, "D. Non-Current Liabilities (D1+D2)", yr)
        E = get_item_year_value(cm, "E. Current Liabilities (E1+E2)", yr)

        STB = get_item_year_value(cm, "1. Short term borrowings", yr)
        if STB is None:
            STB = get_item_year_value(cm, "of which: i) Short term secured loans", yr)

        B2 = get_item_year_value(cm, "2. Inventories", yr)
        B4 = get_item_year_value(cm, "4. Short term investments", yr)
        F1 = get_item_year_value(cm, "1. Sales", yr)
        F5 = get_item_year_value(cm, "5. Other income / (loss)", yr)

        # Share Capital to Number of Shares
        C1 = get_item_year_value(cm, "1. Issued, Subscribed & Paid up capital", yr)
        I2 = (C1 / 10.0) if C1 is not None else None

        ROA = get_item_year_value(cm, "P3. Return on Assets (F7 as a % of Avg {Current year(A+B),previous year (A+B)}", yr)
        if ROA is None:
            ROA = get_item_year_value(cm, "P3. Return on Assets (F7 as a % of Avg {Current year(A+B),previous year (A+B)})", yr)
        if ROA is not None and abs(ROA) > 1:
            ROA = ROA / 100.0

        # --- VARIABLE GENERATION ---
        DR = safe_div(safe_add(D, STB), TotalAB)
        IR = safe_div(B4, TotalAB)
        IncR = safe_div(F5, F1)
        IAR = safe_div(safe_add(A, B2), TotalAB)

        illiquid = safe_add(A, B2)
        NLA_total = None

        if TotalAB is not None and illiquid is not None and D is not None and E is not None:
            NLA_total = TotalAB - illiquid - (D + E)

        NLA = safe_div(NLA_total, I2)

        results.append({
            'Sector': meta['Sector'],
            'Sub-Sector': meta['Sub-Sector'],
            'Company': company,
            'Year': int(yr),
            'ROA': round(ROA, 6) if ROA is not None else "",
            'DR': round(DR, 6) if DR is not None else "",
            'IR': round(IR, 6) if IR is not None else "",
            'IncR': round(IncR, 6) if IncR is not None else "",
            'NLA': round(NLA, 6) if NLA is not None else "",
            'IAR': round(IAR, 6) if IAR is not None else ""
        })

# --- EXPORT ---
out_df = pd.DataFrame(results)

# NO sorting applied here to preserve exact source sequence
out_df.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f"\nSuccess! Created: {out_csv}")
print(f"Total Rows: {len(out_df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Excel file into memory (Sheet: 2005-08)...
Parsing data structure and capturing sector hierarchy...
Computing Shariah-compliance variables in original sequence...

Success! Created: /content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2005_08.csv
Total Rows: 1900


In [7]:
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Paths
file_path = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/2005-23.xlsx'
out_csv = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2009_13.csv'

# --- Helper Functions ---
def normalize_item(s):
    if pd.isna(s): return ""
    return re.sub(r'\s+', ' ', str(s)).strip()

def parse_num(s):
    if pd.isna(s): return None
    t = str(s).strip()
    if t in ("-", "N/A", ""): return None

    neg = False
    if t.startswith('(') and t.endswith(')'):
        neg = True
        t = t[1:-1].strip()

    is_percent = '%' in t
    t = t.replace(',', '').replace('%', '')

    try:
        n = float(t)
        if neg: n = -n
        if is_percent: n = n / 100.0
        return n
    except ValueError:
        return None

def safe_div(a, b):
    if a is None or b is None or pd.isna(a) or pd.isna(b) or b == 0.0:
        return None
    return a / b

def safe_add(*args):
    """Replicates PowerShell's behavior of treating nulls as 0 during addition."""
    valid = [a for a in args if a is not None and not pd.isna(a)]
    if not valid: return None
    return sum(valid)

def get_item_year_value(company_map, item_names, year):
    """Checks a list of possible item names (to handle shifting data schemas) and returns the first match."""
    if isinstance(item_names, str):
        item_names = [item_names]

    for name in item_names:
        if name in company_map and year in company_map[name]:
            return company_map[name][year]

    # Fuzzy fallback for ROA which often changes bracket styles
    if any("Return on Assets" in n for n in item_names):
        for key in company_map.keys():
            if "Return on Assets" in key:
                return company_map[key].get(year, None)

    return None

# --- Execution ---
print("Loading Excel file into memory (Sheet: 2009-13)...")
df = pd.read_excel(file_path, sheet_name='2009-13', header=None)

# Map years to their corresponding zero-indexed column numbers in the 2009-13 sheet
years = { '2009': 4, '2010': 5, '2011': 6, '2012': 7, '2013': 8 }

data = {}
ordered_companies = []  # List to track exact top-to-bottom sequence
company_metadata = {}   # Dictionary to store Sector and Sub-Sector info

print("Parsing data structure and capturing sector hierarchy...")
for index, row in df.iloc[1:].iterrows():
    sector_raw = row[0]     # Col A: Sector
    subsector_raw = row[1]  # Col B: Sub-Sector
    company_raw = row[2]    # Col C: Company Name
    item_raw = row[3]       # Col D: Item Name

    company = str(company_raw).strip() if pd.notna(company_raw) else ""
    item = normalize_item(item_raw)

    if not company or not item: continue

    # Capture company and its metadata on first encounter
    if company not in data:
        data[company] = {}
        ordered_companies.append(company)
        company_metadata[company] = {
            'Sector': str(sector_raw).strip() if pd.notna(sector_raw) else "",
            'Sub-Sector': str(subsector_raw).strip() if pd.notna(subsector_raw) else ""
        }

    if item not in data[company]:
        data[company][item] = {}

    for year_label, col_idx in years.items():
        if col_idx < len(row):
            cell_val = row[col_idx]
            if isinstance(cell_val, (int, float)):
                data[company][item][year_label] = float(cell_val)
            else:
                data[company][item][year_label] = parse_num(cell_val)
        else:
            data[company][item][year_label] = None

print("Computing Shariah-compliance variables in original sequence...")
results = []

# Iterating using the sequence list, NOT alphabetical sort
for company in ordered_companies:
    cm = data[company]
    meta = company_metadata[company]

    for yr in years.keys():

        # --- RAW DATA EXTRACTION (Updated for 2009-13 Schemas) ---
        A = get_item_year_value(cm, ["A. Non-Current Assets (A1+A3+A4+A5+A6)", "A. Non-Current Assets (A2+A3)"], yr)
        B = get_item_year_value(cm, ["B. Current Assets (B1+B2+B3+B4+B5)", "B. Current Assets (B1+B2+B3+B4+B5+B6)"], yr)

        TotalAB = get_item_year_value(cm, "Total Assets (A+B) / Equity & Liabilities (C+D+E)", yr)
        if TotalAB is None:
            TotalAB = safe_add(A, B) if (A is not None or B is not None) else None

        D = get_item_year_value(cm, ["D. Non-Current Liabilities (D1+D2+D3+D4)", "D. Non-Current Liabilities (D1+D2)"], yr)
        E = get_item_year_value(cm, "E. Current Liabilities (E1+E2)", yr)

        STB = get_item_year_value(cm, "1. Short term borrowings", yr)
        if STB is None:
            STB = get_item_year_value(cm, "of which: i) Short term secured loans", yr)

        B2 = get_item_year_value(cm, "2. Inventories", yr)
        B4 = get_item_year_value(cm, "4. Short term investments", yr)
        F1 = get_item_year_value(cm, "1. Sales", yr)
        F5 = get_item_year_value(cm, "5. Other income / (loss)", yr)

        # Share Capital to Number of Shares
        C1 = get_item_year_value(cm, "1. Issued, Subscribed & Paid up capital", yr)
        I2 = (C1 / 10.0) if C1 is not None else None

        ROA = get_item_year_value(cm, "Return on Assets", yr) # Will use the fuzzy matcher
        if ROA is not None and abs(ROA) > 1:
            ROA = ROA / 100.0

        # --- VARIABLE GENERATION ---
        DR = safe_div(safe_add(D, STB), TotalAB)
        IR = safe_div(B4, TotalAB)
        IncR = safe_div(F5, F1)
        IAR = safe_div(safe_add(A, B2), TotalAB)

        illiquid = safe_add(A, B2)
        NLA_total = None

        if TotalAB is not None and illiquid is not None and D is not None and E is not None:
            NLA_total = TotalAB - illiquid - (D + E)

        NLA = safe_div(NLA_total, I2)

        results.append({
            'Sector': meta['Sector'],
            'Sub-Sector': meta['Sub-Sector'],
            'Company': company,
            'Year': int(yr),
            'ROA': round(ROA, 6) if ROA is not None else "",
            'DR': round(DR, 6) if DR is not None else "",
            'IR': round(IR, 6) if IR is not None else "",
            'IncR': round(IncR, 6) if IncR is not None else "",
            'NLA': round(NLA, 6) if NLA is not None else "",
            'IAR': round(IAR, 6) if IAR is not None else ""
        })

# --- EXPORT ---
out_df = pd.DataFrame(results)

# NO sorting applied here to preserve exact source sequence
out_df.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f"\nSuccess! Created: {out_csv}")
print(f"Total Rows: {len(out_df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Excel file into memory (Sheet: 2009-13)...
Parsing data structure and capturing sector hierarchy...
Computing Shariah-compliance variables in original sequence...

Success! Created: /content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2009_13.csv
Total Rows: 2175


In [8]:
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Paths
file_path = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/2005-23.xlsx'
out_csv = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2014_23.csv'

# --- Helper Functions ---
def normalize_item(s):
    if pd.isna(s): return ""
    return re.sub(r'\s+', ' ', str(s)).strip()

def parse_num(s):
    if pd.isna(s): return None
    t = str(s).strip()
    if t in ("-", "N/A", ""): return None

    neg = False
    if t.startswith('(') and t.endswith(')'):
        neg = True
        t = t[1:-1].strip()

    is_percent = '%' in t
    t = t.replace(',', '').replace('%', '')

    try:
        n = float(t)
        if neg: n = -n
        if is_percent: n = n / 100.0
        return n
    except ValueError:
        return None

def safe_div(a, b):
    if a is None or b is None or pd.isna(a) or pd.isna(b) or b == 0.0:
        return None
    return a / b

def safe_add(*args):
    """Replicates PowerShell's behavior of treating nulls as 0 during addition."""
    valid = [a for a in args if a is not None and not pd.isna(a)]
    if not valid: return None
    return sum(valid)

def get_item_year_value(company_map, item_names, year):
    """Upgraded Fuzzy Matcher: Automatically finds variables even if SBP changes the (brackets)."""
    if isinstance(item_names, str):
        item_names = [item_names]

    # 1. Try exact matches first
    for name in item_names:
        if name in company_map and year in company_map[name]:
            return company_map[name][year]

    # 2. Try partial matching (e.g. finding "D. Non-Current Liabilities" inside "D. Non-Current Liabilities (D1+D2+D3+D4+D5)")
    for name in item_names:
        for key in company_map.keys():
            if str(name).lower() in str(key).lower():
                if year in company_map[key] and company_map[key][year] is not None:
                    return company_map[key][year]

    return None

# --- Execution ---
print("Loading Excel file into memory (Sheet: 2014-23)...")
df = pd.read_excel(file_path, sheet_name='2014-23', header=None)

# Map years to their corresponding zero-indexed column numbers in the 2014-23 sheet
years = {
    '2014': 4, '2015': 5, '2016': 6, '2017': 7, '2018': 8,
    '2019': 9, '2020': 10, '2021': 11, '2022': 12, '2023': 13
}

data = {}
ordered_companies = []  # List to track exact top-to-bottom sequence
company_metadata = {}   # Dictionary to store Sector and Sub-Sector info

print("Parsing data structure and capturing sector hierarchy...")
for index, row in df.iloc[1:].iterrows():
    sector_raw = row[0]     # Col A: Sector
    subsector_raw = row[1]  # Col B: Sub-Sector
    company_raw = row[2]    # Col C: Company Name
    item_raw = row[3]       # Col D: Item Name

    company = str(company_raw).strip() if pd.notna(company_raw) else ""
    item = normalize_item(item_raw)

    if not company or not item: continue

    # Capture company and its metadata on first encounter
    if company not in data:
        data[company] = {}
        ordered_companies.append(company)
        company_metadata[company] = {
            'Sector': str(sector_raw).strip() if pd.notna(sector_raw) else "",
            'Sub-Sector': str(subsector_raw).strip() if pd.notna(subsector_raw) else ""
        }

    if item not in data[company]:
        data[company][item] = {}

    for year_label, col_idx in years.items():
        if col_idx < len(row):
            cell_val = row[col_idx]
            if isinstance(cell_val, (int, float)):
                data[company][item][year_label] = float(cell_val)
            else:
                data[company][item][year_label] = parse_num(cell_val)
        else:
            data[company][item][year_label] = None

print("Computing Shariah-compliance variables in original sequence...")
results = []

# Iterating using the sequence list, NOT alphabetical sort
for company in ordered_companies:
    cm = data[company]
    meta = company_metadata[company]

    for yr in years.keys():

        # --- SMART RAW DATA EXTRACTION ---
        # The fuzzy matcher will automatically find the correct line item regardless of how the brackets are formatted
        A = get_item_year_value(cm, "A. Non-Current Assets", yr)
        B = get_item_year_value(cm, "B. Current Assets", yr)

        TotalAB = get_item_year_value(cm, "Total Assets", yr)
        if TotalAB is None:
            TotalAB = safe_add(A, B) if (A is not None or B is not None) else None

        D = get_item_year_value(cm, "D. Non-Current Liabilities", yr)
        E = get_item_year_value(cm, "E. Current Liabilities", yr)

        STB = get_item_year_value(cm, ["1. Short term borrowings", "Short term borrowings", "Short term secured loans"], yr)
        B2 = get_item_year_value(cm, ["2. Inventories", "Inventories"], yr)
        B4 = get_item_year_value(cm, ["4. Short term investments", "Short term investments"], yr)

        F1 = get_item_year_value(cm, ["1. Sales", "Sales"], yr)
        F5 = get_item_year_value(cm, ["5. Other income", "Other income"], yr)

        # Share Capital to Number of Shares
        C1 = get_item_year_value(cm, ["1. Issued, Subscribed & Paid up capital", "Issued, Subscribed & Paid up capital"], yr)
        I2 = (C1 / 10.0) if C1 is not None else None

        ROA = get_item_year_value(cm, "Return on Assets", yr)
        if ROA is not None and abs(ROA) > 1:
            ROA = ROA / 100.0

        # --- VARIABLE GENERATION ---
        DR = safe_div(safe_add(D, STB), TotalAB)
        IR = safe_div(B4, TotalAB)
        IncR = safe_div(F5, F1)
        IAR = safe_div(safe_add(A, B2), TotalAB)

        illiquid = safe_add(A, B2)
        NLA_total = None

        if TotalAB is not None and illiquid is not None and D is not None and E is not None:
            NLA_total = TotalAB - illiquid - (D + E)

        NLA = safe_div(NLA_total, I2)

        results.append({
            'Sector': meta['Sector'],
            'Sub-Sector': meta['Sub-Sector'],
            'Company': company,
            'Year': int(yr),
            'ROA': round(ROA, 6) if ROA is not None else "",
            'DR': round(DR, 6) if DR is not None else "",
            'IR': round(IR, 6) if IR is not None else "",
            'IncR': round(IncR, 6) if IncR is not None else "",
            'NLA': round(NLA, 6) if NLA is not None else "",
            'IAR': round(IAR, 6) if IAR is not None else ""
        })

# --- EXPORT ---
out_df = pd.DataFrame(results)

# NO sorting applied here to preserve exact source sequence
out_df.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f"\nSuccess! Created: {out_csv}")
print(f"Total Rows: {len(out_df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Excel file into memory (Sheet: 2014-23)...
Parsing data structure and capturing sector hierarchy...
Computing Shariah-compliance variables in original sequence...

Success! Created: /content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2014_23.csv
Total Rows: 4510


In [9]:
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. File Paths
base_path = '/content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/'

file1 = base_path + 'computed_variables_2005_08.csv'
file2 = base_path + 'computed_variables_2009_13.csv'
file3 = base_path + 'computed_variables_2014_23.csv'
out_file = base_path + 'computed_variables_2005_23_Combined.csv'

print("Loading the three CSV files...")
# Load the dataframes
try:
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)
    df3 = pd.read_csv(file3)
except FileNotFoundError as e:
    print(f"Error finding file: {e}")
    print("Please make sure all three scripts have been successfully run first.")
    raise

# 3. Combine all rows into one massive dataset
print("Concatenating files...")
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

# 4. Extract the exact original top-to-bottom sequence of companies from File 1
# drop_duplicates() preserves the first time it sees each company, keeping your Excel sequence intact
original_company_sequence = df1['Company'].drop_duplicates().tolist()

# (Safety Check): If any new companies appeared in 2009-13 or 2014-23 that weren't in 2005-08,
# append them to the end of our sequence list so they aren't lost.
all_unique_companies = df_combined['Company'].drop_duplicates().tolist()
for company in all_unique_companies:
    if company not in original_company_sequence:
        original_company_sequence.append(company)

print("Sorting by Original Company Sequence, then chronologically by Year...")
# 5. Convert the 'Company' column into a Categorical data type with our custom sequence
df_combined['Company'] = pd.Categorical(
    df_combined['Company'],
    categories=original_company_sequence,
    ordered=True
)

# 6. Sort! Because 'Company' is categorical, it will sort by your Excel order, then by Year
df_combined.sort_values(by=['Company', 'Year'], inplace=True)

# 7. Export the final combined file
df_combined.to_csv(out_file, index=False, encoding='utf-8-sig')

print(f"\nSuccess! Created Final Master File: {out_file}")
print(f"Total Rows: {len(df_combined)}")

# Display a quick preview of the top 20 rows to verify Company #1 has all its years stacked
print("\nPreview of the top 20 rows:")
display(df_combined.head(20))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading the three CSV files...
Concatenating files...
Sorting by Original Company Sequence, then chronologically by Year...

Success! Created Final Master File: /content/drive/MyDrive/Github-Cursor/Shariah Compliance Project/computed_variables_2005_23_Combined.csv
Total Rows: 8585

Preview of the top 20 rows:


,Sector,Sub-Sector,Company,Year,ROA,DR,IR,IncR,NLA,IAR
0,All Sector,All Sector,703 - All Sector,2005,0.143694,0.132304,0.065360,0.018623,-7.642726,0.628045
1,All Sector,All Sector,703 - All Sector,2006,0.135754,0.136683,0.092916,0.024840,-10.109068,0.618097
2,All Sector,All Sector,703 - All Sector,2007,0.102382,0.146254,0.099476,0.026431,-11.582066,0.615870
3,All Sector,All Sector,703 - All Sector,2008,0.076706,0.146681,0.086745,0.021596,-16.623123,0.607348
1900,All Sector,All Sector,703 - All Sector,2009,0.065554,0.398825,0.022010,0.023306,-26.652790,0.690544
1901,All Sector,All Sector,703 - All Sector,2010,0.083029,0.371054,0.023401,0.023231,-25.612059,0.672492
1902,All Sector,All Sector,703 - All Sector,2011,0.085782,0.365054,0.025476,0.022464,-25.257168,0.666431
1903,All Sector,All Sector,703 - All Sector,2012,0.102173,0.343040,0.024855,0.023066,-23.509525,0.610004
1904,All Sector,All Sector,703 - All Sector,2013,0.096918,0.327521,0.030883,0.023274,-26.093404,0.685184
4075,All Sector,All Sector,703 - All Sector,2014,0.081570,0.302567,0.036447,0.024242,-24.302623,0.647854
